# Calibration peak-centroid and held-out validation

This notebook audits the working manual-primary energy calibration used in the v0.1 snapshot. It reports counting-statistical centroid uncertainties, leave-one-peak-out checks, and deliberately limited diagnostics for the broad four-flight feature near 511 keV and the threshold-adjacent low-energy structure.

## Interpretation boundary

The bootstrap intervals cover Poisson counting statistics for fixed windows and a fixed sideband method. They do not include line blending, source characterization, detector response, drift, background/window choice, or calibration-model uncertainty. A detector-spectrum centroid near 511 keV is not particle or isotope identification. The low-energy structure is below the lowest 59.5-keV anchor, so no precision centroid is reported for it.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'calibration' / 'validate_peak_centroids.py').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'calibration' / 'validate_peak_centroids.py').is_file(), ROOT
ROOT

In [ ]:
check = subprocess.run(
    [sys.executable, 'calibration/validate_peak_centroids.py', '--check'],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(check.stdout.strip())

In [ ]:
validation = ROOT / 'calibration' / 'validation'
anchors = pd.read_csv(validation / 'anchor_centroid_uncertainties.csv')
held_out = pd.read_csv(validation / 'held_out_peak_validation.csv')
features = pd.read_csv(validation / 'flight_feature_centroid_diagnostics.csv')
print(anchors[[
    'nuclide_or_line', 'assigned_energy_keV',
    'centroid_counting_stat_1sigma_channel',
    'centroid_counting_stat_1sigma_keV'
]].to_string(index=False))

In [ ]:
nearby = held_out[held_out['near_511_context'].eq('yes')]
print(nearby[[
    'held_out_line', 'assigned_energy_keV', 'predicted_energy_keV',
    'residual_predicted_minus_assigned_keV',
    'counting_stat_prediction_1sigma_keV'
]].to_string(index=False))

In [ ]:
print(features.to_string(index=False))

## Release interpretation

The four-flight broad feature has a descriptive fitted centroid of 510.99 keV with an approximate counting-statistical 68% profile interval of 508.54–513.31 keV. This raw-channel aggregation uses one detector and assumes stable response across the four acquisition dates. Nearby held-out folds at 583.187 and 609.312 keV have residuals of +3.93 and -4.36 keV, respectively; those discrepancies are larger than the centroid's counting-statistical error and remain only one part of the systematic uncertainty.

The threshold-adjacent low-energy structure has no reported precision centroid. Its per-flight channel maxima map to 14.88–19.75 keV and simple fit variants span 15.67–19.93 keV, all below the lowest calibration anchor. Controlled reference data below 59.5 keV are required before a stronger line-energy claim.